In [1]:
IMAGE_PATH_BASE = '11Classes'
INPUT_FILE_PATH = './11Classes.csv'

In [ ]:
# Export FinalDataset to csv

import pandas as pd
import os
from PIL import Image
from tqdm.notebook import tqdm
import numpy as np
from PIL import Image
import os
from glob import glob
from tqdm.notebook import tqdm

def downscale(image : Image, size):
    image.thumbnail((size, size), Image.ANTIALIAS)
    return image

def crop(image : Image, size : int):
    width, height = image.size
    
    left = (width - size) / 2
    top = (height - size) / 2
    right = (width + size) / 2
    bottom = (height + size) / 2
    
    return image.crop((left, top, right, bottom))

result_dataset_path = f"{IMAGE_PATH_BASE}"
resized_images_base_path = "../Step 2 - Image dataset/Images"

if not os.path.exists(result_dataset_path):
    os.mkdir(result_dataset_path) 

dataset = pd.read_csv(f"{INPUT_FILE_PATH}", dtype={
    "Id": "string",
    "Class": "string",
    'OriginalDatasetId': "string"})

empty_galaxy_count = 0
for _, row in tqdm(dataset.iterrows(), total = len(dataset)):
    written = False

    if not os.path.exists(f"{result_dataset_path}/{row['Id']}"):
        os.mkdir(f"{result_dataset_path}/{row['Id']}")

    if os.path.exists(f"{resized_images_base_path}/GZ12_All_Resized/{row['OriginalDatasetId']}.jpg"):
        gz12_img = Image.open(f"{resized_images_base_path}/GZ12_All_Resized/{row['OriginalDatasetId']}.jpg")
        gz12_img.save(f"{result_dataset_path}/{row['Id']}/{row['OriginalDatasetId']}.jpg")
        written = True
    elif os.path.exists(f"{resized_images_base_path}/GZ12_All/{row['OriginalDatasetId']}.jpg"):
        img = Image.open(f"{resized_images_base_path}/GZ12_All/{row['OriginalDatasetId']}.jpg")
        img = crop(img, 224)
        img = downscale(img, 128)

        # img.save(f"{resized_images_base_path}/GZ12_All/{row['OriginalDatasetId']}.jpg")
        img.save(f"{result_dataset_path}/{row['Id']}/{row['OriginalDatasetId']}.jpg")
        written = True
    else:
        print(row['OriginalDatasetId'])

    if not written:
        empty_galaxy_count = empty_galaxy_count + 1

print(empty_galaxy_count)

In [4]:
import tensorflow as tf

IMAGE_SIZE = 128 #@param
CLASSES = [b"Er", b"Ei", b"Ec", b"Sa", b"Sb", b"Sc", b"Sd", b"SBa", b"SBb", b"SBc", b"SBd"] # 11 classes
AUTO = tf.data.experimental.AUTOTUNE

def decode_jpeg_and_label(item):
  filename = item['image_path']
  galaxy_class = item['class']
  objid = item['objid']
  ids = item['id']
  
  bits = tf.io.read_file(filename)
  image = tf.image.decode_jpeg(bits)
  return image, galaxy_class, objid, ids

def recompress_image(image, label, objid, ids):
  image = tf.image.encode_jpeg(image, optimize_size=True, chroma_downsampling=False)
  return image, label, objid, ids

def _bytestring_feature(list_of_bytestrings):
  return tf.train.Feature(bytes_list=tf.train.BytesList(value=list_of_bytestrings))

def _int_feature(list_of_ints): # int64
  return tf.train.Feature(int64_list=tf.train.Int64List(value=list_of_ints))

def _float_feature(list_of_floats): # float32
  return tf.train.Feature(float_list=tf.train.FloatList(value=list_of_floats))

def to_tfrecord(tfrec_filewriter, img_bytes, label, objid, id):  
  class_num = np.argmax(np.array(CLASSES)==label) # 'roses' => 2 (order defined in CLASSES)
  one_hot_class = np.eye(len(CLASSES))[class_num]     # [0, 0, 1, 0, 0] for class #2, roses

  feature = {
      "image": _bytestring_feature([img_bytes]), # one image in the list
      "class": _int_feature([class_num]),        # one class in the list
      "label":         _bytestring_feature([label]),          # fixed length (1) list of strings, the text label
      "objid":         _bytestring_feature([objid]),          # fixed length (1) list of strings, the text label
      "id":         _bytestring_feature([objid]),          # fixed length (1) list of strings, the text label
      "one_hot_class": _float_feature(one_hot_class.tolist()) # variable length  list of floats, n=len(CLASSES)
  }
  return tf.train.Example(features=tf.train.Features(feature=feature))

def create_tf_record_dataset(filenames, items_per_record, output_folder):
  dataset2 = filenames.map(decode_jpeg_and_label, num_parallel_calls=AUTO)
  dataset3 = dataset2.map(recompress_image, num_parallel_calls=AUTO)
  dataset3 = dataset3.batch(items_per_record) # sharding: there will be one "batch" of images per file

  print("Writing TFRecords")
  for shard, (image, label, ids, objid) in enumerate(dataset3):
    # batch size used as shard size here
    shard_size = image.numpy().shape[0]
    filename = output_folder + "/{:02d}-{}.tfrec".format(shard, shard_size)
    
    np_image = image.numpy()
    np_label = label.numpy()
    np_objid = objid.numpy()
    np_id = ids.numpy()

    with tf.io.TFRecordWriter(filename) as out_file:
      for i in range(shard_size):
        example = to_tfrecord(out_file,
                              np_image[i], # re-compressed image: already a byte string
                              np_label[i],
                              np_objid[i],
                              np_id)
        out_file.write(example.SerializeToString())
      print("Wrote file {} containing {} records".format(filename, shard_size))
  print("Done.")

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import tensorflow as tf
import pandas as pd
import os
import math


ids_to_path = lambda ids: np.array([f"{IMAGE_PATH_BASE}/{row[0]}/{row[1]}.jpg" for row in ids])

def convert_ids_and_classes_to_dataset(ids, classes):
    print(ids[:, 1])
    dataset_dictionary = {
        "image_path": ids_to_path(ids),
        "class": classes.T,
        "id": ids[:, 0].T.astype(str),
        "objid": ids[:, 1].T.astype(str)
    }

    return tf.data.Dataset.from_tensor_slices(dataset_dictionary)



def create_records_for_training_and_test_images(training_data, test_data, path):

    if not os.path.exists(f"{path}/train"):
        os.makedirs(f"{path}/train") 
        os.makedirs(f"{path}/test") 

    number_of_galaxies_to_pad = BATCH_SIZE - len(training_data) % BATCH_SIZE

    resample_indexes = np.random.choice(
        np.arange(len(training_data)),
        number_of_galaxies_to_pad,
        replace=False)

    original_train_ids = training_data[[f"Id", "OriginalDatasetId"]]
    original_train_classes = training_data[f"Class"]

    padded_train_ids = np.concatenate((original_train_ids, original_train_ids.iloc[resample_indexes]))
    padded_train_classes = np.concatenate((original_train_classes, original_train_classes.iloc[resample_indexes]))

    test_ids = test_data[["Id", "OriginalDatasetId"]]
    test_classes = test_data["Class"]

    tf_dataset = convert_ids_and_classes_to_dataset(padded_train_ids, padded_train_classes)
    create_tf_record_dataset(tf_dataset, 4096, f'{path}/train')

    tf_dataset = convert_ids_and_classes_to_dataset(test_ids.to_numpy(), test_classes.to_numpy())
    create_tf_record_dataset(tf_dataset, 4096, f'{path}/test')



BATCH_SIZE = 1024
SEED = 17
N_FOLDS = 10

np.random.seed(SEED)

print("Loading dataset")

dataset = pd.read_csv(f"{INPUT_FILE_PATH}", dtype={
    'OriginalDatasetId': "string"})

dataset["ImageExists"] = dataset["OriginalDatasetId"] != '587731174917669027'
dataset = dataset[dataset["ImageExists"]]

print("=====Overall====")
print(dataset["Class"].value_counts())


# Take 10% of galaxies for tuning
tuning_galaxies = dataset.sample(n= len(dataset) // 10, replace=False, random_state= SEED)
print("=====Tuning====")
print(tuning_galaxies["Class"].value_counts())

# Take remaining 90% galaxies for training 
training_galaxies = dataset[~(dataset["Id"].isin(tuning_galaxies["Id"]))]
print("=====Training====")
print(training_galaxies["Class"].value_counts())


if not os.path.exists(f"TFDataset/{IMAGE_PATH_BASE}"):
    os.mkdir(f"TFDataset/{IMAGE_PATH_BASE}") 


# -------------------------- Tuning data preparation -----------------------------------

tuning_test_galaxies = tuning_galaxies.sample(n= len(tuning_galaxies) // 10, replace=False, random_state= SEED)
tuning_train_galaxies = tuning_galaxies[~(tuning_galaxies["Id"].isin(tuning_test_galaxies["Id"]))]

create_records_for_training_and_test_images(tuning_train_galaxies, tuning_test_galaxies, f"TFDataset/{IMAGE_PATH_BASE}/tuning")


# -------------------------- Training K-fold data preparation -----------------------------------

kfold = StratifiedKFold(n_splits = N_FOLDS, shuffle = True, random_state = SEED)

fold = 1



for train_index, test_index in kfold.split(training_galaxies[["Id", "OriginalDatasetId"]], training_galaxies["Class"] ):
    print("fold", fold)

    dataset_folder_base = f"TFDataset/{IMAGE_PATH_BASE}/training/fold_{fold}"

    # Still working with GALAXIES here, ensuring that images of one galaxy are only in training or in test split, they will not mix.
    training_data = training_galaxies.iloc[train_index]
    test_data = training_galaxies.iloc[test_index]
    
    create_records_for_training_and_test_images(training_data, test_data, dataset_folder_base)

    fold += 1
